<a href="https://colab.research.google.com/github/phi1z/1yanagiLab/blob/main/MPI_Simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Langevin magnetization

## Langevin Magnetization

The magnetization of single domain nanoparticles are represented by Langevin function, it follows:
\begin{equation}
  M(H) = M_s \left( \frac{1}{\tanh(\xi)} - \frac{1}{\xi} \right), \quad \left(  \xi = \frac{\mu H}{k_\text{B}T} = \frac{3\chi_0}{M_s}H \right),
\end{equation}
where $M_s$ is the satulation magnetization, $\chi_0$ is the initial susceptability, and $H$ is field.

## Debye relaxation model

In AC magnetic field (AMF), the magnetization are relax and its the magnitude become weeker, the larger frequency $f$ is.
Thus, susceptability $\chi$ needs to rewrite as AC one, and $\chi$ would be complex number like Impedance.
In Debye model AC susceptability $\chi^\text{(AC)}$ are
\begin{equation}
  \chi^\text{(AC)} = \frac{\chi_0}{1 - i2\pi f\tau_\text{eff}}
  = \frac{1}{1 + (2\pi f\tau_\text{eff})^2}\chi_0 + i\frac{2\pi f\tau_\text{eff}}{1 + (2\pi f\tau_\text{eff})^2}\chi_0
  = \chi' + i\chi'',
\end{equation}
where $\tau_\text{eff}$ is effective relaxation time.

Whwn AMF $H(t) = H_0\cos(2\pi f t)$ apply, the AC magnetization $M(t)$ would be:
\begin{equation}
  M(t) = \chi^\text{(AC)}H(t) = \chi'H_0\cos(2\pi f t) - \chi''H_0\sin(2\pi f t).
\end{equation}
To use trigonometric function composition, $M(t)$ would be
\begin{equation}
  M(t) = \sqrt{{\chi'}^2 + {\chi''}^2}H_0\cos(2\pi ft + \phi), \quad
  \left( \phi = \tan^{-1}(\chi''/\chi') = \tan^{-1}(2\pi f\tau_\text{eff}) \right).
\end{equation}
So AC magnetization $M(t)$ delay in $\phi$ degrees.

By the way, power $\sqrt{{\chi'}^2 + {\chi''}^2}$ can be written
\begin{equation}
  \sqrt{{\chi'}^2 + {\chi''}^2} = \frac{\chi_0}{\sqrt{1 + (2\pi f\tau_\text{eff})^2}}.
\end{equation}

## Two type relaxation

### Neel relaxation
Neel relaxation is one of the relaxation mechanisms, which involves rotating only magnetic moments.
The relaxation time $\tau_\text{N}$ may be described by
\begin{equation}
  \tau_\text{N} = \tau_0 \exp\left( \frac{K_\text{eff}V_\text{M}}{k_\text{B}T} \right),
\end{equation}
where $\tau_0$ is attempt time, $K_\text{eff}$ is anisotropy constant, $V_\text{M}$ is volume of the nanocages, $k_\text{B}$ is Boltzmann constant, and $T$ is temperature.
$\tau_\text{N}$ is dependent on the material, size, or shape of the nanocage.

### Brownian relaxation
Brownian relaxation is one of the relaxation mechanisms, which involves rotating the material itself.
The relaxation time $\tau_\text{B}$ may be described by
\begin{equation}
  \tau_\text{B} = \frac{3\eta V_\text{H}}{k_\text{B}T},
\end{equation}
where $\eta$ is the viscosity of the solvent and $V_\text{H}$ is the hydrodynamic volume of nanocages.
$\tau_\text{B}$ is dependent on the particle features in the liquid.

### Effecctive relaxation
Neel and Brownian relaxations occur in parallel, with an effective relaxation time as an inverse of effective relaxation time, given by

\begin{equation}
  \frac{1}{\tau_\text{eff}} = \frac{1}{\tau_\text{N}} + \frac{1}{\tau_\text{B}}
  \quad \Longleftrightarrow \quad
  \tau_\text{eff} = \frac{\tau_\text{N}\tau_\text{B}}{\tau_\text{N} + \tau_\text{B}}.
\end{equation}

## AC Langevin magnetization

To summarize all equations, AC Magetization of nanoparticle represent by:

\begin{equation}
  M(t) \simeq \frac{M_s}{\sqrt{1 + (2\pi f\tau_\text{eff})^2}} \left( \frac{1}{\tanh(\xi)} - \frac{1}{\xi} \right), \quad \left( \xi = \frac{3\chi_0H_0}{M_s}\cos(2\pi ft + \phi) \right),
\end{equation}

## MPI signal

3rd harmonics amplitude is used as MIP signal.
$n$-th harmoniccs amplitude $\tilde{A}_n$ are calcurate by discrete Fourie transformation (DFT),
\begin{equation}
  \tilde{A}_n = \int_0^{1/f} e^{2n\pi ft} M(t)\, dt.
  = \frac{2}{N_T}\sum_{n=1}^{N_T} e^{2n\pi ft} M(t_i)
\end{equation}
In mesuearment, look-in-amp. can read only power $A_n$,
\begin{equation}
  A_n = \sqrt{\Re[\tilde{A}_n]^2 + \Im[\tilde{A}_n]^2}.
\end{equation}
$A_3$ is the 3rd harmonics amplitude.



In [1]:
#@title import AC Langevin Magnetization Model {run: "auto"}

import matplotlib.pyplot as plt
import numpy as np

def SI_cgs(x, type="M"):
  if type == "M":
    return x * 1.0e-3
  elif type == "H":
    return x * 4 * np.pi * 1.0e-3

def cgs_SI(x, type="M"):
  if type == "M":
    return x * 1.0e3
  elif type == "H":
    return x * 1.0e3 / (4 * np.pi)

class LangevinMagnetization:
    def __init__(self, M_s, chi_0):
        if M_s <= 0:
            raise ValueError("M_s must be positive.")
        if chi_0 <= 0:
            raise ValueError("chi_0 must be positive.")

        self.Ms = M_s
        self.chi = chi_0
        self.mkb = 3 * self.chi / self.Ms

    def langevin_function(self, x):
        if x == 0:
            return 0.0
        return 1 / np.tanh(x) - 1 / x

    def calculate_magnetization(self, H_field):
        x_values = self.mkb * H_field

        langevin_vec = np.vectorize(self.langevin_function)
        l_values = langevin_vec(x_values)

        magnetization = self.Ms * l_values
        return magnetization

    def plot_magnetization_curve(self, H_max=50000, num_points=1024, unit="SI"):
        H_fields = np.linspace(-1 * H_max, H_max, num_points)
        magnetizations = self.calculate_magnetization(H_fields)

        plt.figure(figsize=(8, 6))
        if unit == "SI":
            plt.plot(H_fields * 1.0e-3, magnetizations * 1.0e-3,
                    label=r'$M_s$={:.3g} kA/m, $\chi_0$={:.2g}'.format(self.Ms*1.0e-3, self.chi))
            plt.xlabel('Magnetic Field H (kA/m)', fontsize=16)
            plt.ylabel('Magnetization M (kA/m)', fontsize=16)
        else:
            plt.plot(SI_cgs(H_fields, "H"), SI_cgs(magnetizations, "M"),
                    label=r'$M_s$={:.3g} emu/cm$^3$, $\chi_0$={:.2g}'.format(SI_cgs(self.Ms, "M"), self.chi))
            plt.xlabel('Magnetic Field H (Oe)', fontsize=16)
            plt.ylabel(f'Magnetization M (emu/cm$^3$)', fontsize=16)

        plt.title('Langevin Magnetization Curve')
        plt.grid(True)
        plt.legend(fontsize=14)
        plt.show()

class AC_LangevinMagnetization(LangevinMagnetization):
    def __init__(self, M_s, chi_0, frequency, relaxation_time):
        super().__init__(M_s, chi_0)

        if frequency <= 0:
            raise ValueError("frequancy must be positive.")
        if relaxation_time <= 0:
            raise ValueError("relaxation time must be positive.")

        self.frequency = frequency
        self.relaxation_time = relaxation_time

    def calculate_ac_magnetization(self, H_max, loop=1, n=1024, text=True):
        t_max = 1.0 / self.frequency * loop * 1.1
        t = np.linspace(0, t_max, n)
        H_t = H_max * np.sin(2 * np.pi * self.frequency * t)
        y = 2 * np.pi * self.frequency * self.relaxation_time
        chip = 1 / (1 + y**2)
        chipp = y / (1 + y**2)
        phi = np.arctan2(chipp, chip)
        if text:
            print(f"phi = {phi*180/np.pi:.3g}deg, y = {y:.3g}")
        im_H_t = H_max * np.sin(2 * np.pi * self.frequency * t + phi)
        M_t = super().calculate_magnetization(im_H_t) / (1 + y**2)
        return t, H_t, M_t

    def calculate_ac_voltage(self, t, H_t, M_t, rate=1.2566e-6, plot=False):
        B = rate * (H_t + M_t)
        V_t = np.gradient(B, t)
        if plot:
          fig, ax = plt.subplots(figsize=(8, 6))
          ax.plot(t, B, label='B(t)', color="black")
          ax2 = ax.twinx()
          ax2.plot(t, V_t, label='V(t)', color="red")
          ax.set_xlabel('Time (s)', fontsize=14)
          ax.set_ylabel('Magnetic Flux, B (T)', fontsize=14)
          ax2.set_ylabel('Voltage (V)', fontsize=14, color="red")
          ax.set_title('Voltage vs Time', fontsize=16)
          ax.grid(True)
          plt.show()
        return V_t

    def plot_ac_magnetization_curve(self, t, H_t, M_t, unit="SI"):
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        # Determine units and labels
        if unit == "SI":
            h_label = 'Magnetic Field H (kA/m)'
            m_label = 'Magnetization M (kA/m)'
            h_scale = 1.0e-3
            m_scale = 1.0e-3
        else: # cgs
            h_label = 'Magnetic Field H (Oe)'
            m_label = 'Magnetization M (emu/cm$^3$)'
            h_t_cgs = SI_cgs(H_t, "H")
            M_t_cgs = SI_cgs(M_t, "M")
            h_scale = 1.0
            m_scale = 1.0

        # Plot 1: H(t) and M(t) vs time
        ax1 = axes[0]
        ax1_twin = ax1.twinx()

        if unit == "SI":
            ax1.plot(t, H_t * h_scale, 'b-', label='Magnetic Field H')
            ax1_twin.plot(t, M_t * m_scale, 'r-', label='Magnetization M')
        else:
            ax1.plot(t, h_t_cgs * h_scale, 'b-', label='Magnetic Field H')
            ax1_twin.plot(t, M_t_cgs * m_scale, 'r-', label='Magnetization M')

        ax1.set_xlabel('Time (s)', fontsize=14)
        ax1.set_ylabel(h_label, color='b', fontsize=14)
        ax1_twin.set_ylabel(m_label, color='r', fontsize=14)
        ax1.set_title('Magnetic Field and Magnetization vs Time', fontsize=16)
        ax1.grid(True)

        # Combine legends from both axes
        lines, labels = ax1.get_legend_handles_labels()
        lines2, labels2 = ax1_twin.get_legend_handles_labels()
        ax1_twin.legend(lines + lines2, labels + labels2, loc='best', fontsize=12)

        # Plot 2: M-H loop
        ax2 = axes[1]
        if unit == "SI":
            ax2.plot(H_t * h_scale, M_t * m_scale, 'g-')
        else:
            ax2.plot(h_t_cgs * h_scale, M_t_cgs * m_scale, 'g-')

        ax2.set_xlabel(h_label, fontsize=14)
        ax2.set_ylabel(m_label, fontsize=14)
        ax2.set_title('M-H Loop (Langevin Magnetization)', fontsize=16)
        ax2.grid(True)

        plt.tight_layout()
        plt.show()


class DFTMagnetization(AC_LangevinMagnetization):
    def __init__(self, M_s, chi_0, frequency, relaxation_time):
        super().__init__(M_s, chi_0, frequency, relaxation_time)

    def raw_fft(self, t, V_t):
        N = len(V_t)
        dt = t[1] - t[0]
        if dt <= 0:
            raise ValueError("Sampling interval dt must be positive.")

        # 2. Apply Hanning window to M_t data
        window = np.hanning(N)
        V_t_windowed = V_t * window

        # 3. Perform Fast Fourier Transform (FFT) on the windowed M_t data
        V_fft = np.fft.fft(V_t_windowed)
        freqs = np.fft.fftfreq(N, dt)

        # 5. Keep only the positive frequencies and their corresponding FFT amplitudes (single-sided spectrum)
        positive_freq_indices = np.where(freqs > 0)
        freqs_positive = freqs[positive_freq_indices]
        V_fft_positive = V_fft[positive_freq_indices]

        # Normalize amplitudes for single-sided spectrum.
        amplitudes_positive = 2 * np.abs(V_fft_positive) / N
        return freqs_positive, amplitudes_positive

    def calculate_harmonics(self, t, V_t, num_harmonics=10):

        freqs_positive, amplitudes_positive = self.raw_fft(t, V_t)
        # 6. Extract the amplitudes of the harmonic components
        harmonic_orders = []
        harmonic_amplitudes = []
        fundamental_frequency = self.frequency

        for order in range(1, num_harmonics + 1):
            target_freq = order * fundamental_frequency

            # Ensure the target frequency is within our positive frequency range
            if target_freq > freqs_positive.max():
                break # Stop if target frequency exceeds max available frequency

            # Find the index in freqs_positive closest to the target_freq
            idx = np.argmin(np.abs(freqs_positive - target_freq))

            harmonic_orders.append(order)
            harmonic_amplitudes.append(amplitudes_positive[idx])

        return np.array(harmonic_orders), np.array(harmonic_amplitudes)

    def plot_harmonics(self, harmonic_orders, harmonic_amplitudes, rid1=True):

        y_label = r'Amplitude, $A_n$'
        d_start_range = 0
        if rid1:
            d_start_range = 1
            y_label = r'Amplitude, $A_n / A_1$'
            harmonic_amplitudes = harmonic_amplitudes.copy() / harmonic_amplitudes[0]
        plt.figure(figsize=(10, 6))
        plt.bar(harmonic_orders[d_start_range:], harmonic_amplitudes[d_start_range:], color='skyblue')
        plt.xlabel('Harmonic Order, $n$', fontsize=14)
        plt.ylabel(y_label, fontsize=14)
        plt.yscale('log')
        plt.title('Harmonic Amplitudes of Voltage', fontsize=16)
        plt.xticks(harmonic_orders[d_start_range:]) # Ensure integer ticks for harmonic orders
        plt.grid(True)
        plt.show()

def calculate_brown_relaxation_time(eta, D, T, rate=1.0):
    # Boltzmann constant in J/K
    k_B = 1.380649e-23

    # Calculate Brown relaxation time (tau_B)
    V_H = np.pi * D**3 / 6
    tau_B = (3 * eta * V_H) / (k_B * T)
    return tau_B * rate

def calculate_neel_relaxation_time(K, d, T, tau_0=1e-9):
    # Boltzmann constant in J/K
    k_B = 1.380649e-23
    # Calculate Neel relaxation time (tau_N)
    V_M = np.pi * d**3 / 6
    tau_N = tau_0 * np.exp((K * V_M) / (k_B * T))
    return tau_N

def calculate_effective_relaxation_time(tau_B, tau_N):
    # Calculate effective relaxation time (tau_eff) from tau_B and tau_N
    # The formula is 1/tau_eff = 1/tau_B + 1/tau_N, which simplifies to tau_eff = (tau_B * tau_N) / (tau_B + tau_N)
    if tau_B <= 0 or tau_N <= 0:
        raise ValueError("Brown and Neel relaxation times must be positive.")
    tau_eff = (tau_B * tau_N) / (tau_B + tau_N)
    return tau_eff

print("Langevin relaxation model was succesfully constructed!")


Langevin relaxation model was succesfully constructed!


In [ ]:
#@title AC Langevin Magnetization  {run: "auto"}

# @markdown Set unit and material:
density = 5.17 #@param {"type": "number"}
unit = "cgs" #@param ["SI", "cgs"]
# @markdown Select which relaxation time to use:
relaxation_time_choice = "Brown" #@param ["Brown", "Neel", "Effective"]

# @markdown Set Particle Parameter
eta = 30 #@param {type:"number"}
eta = eta * 1.0e-3
D = 60 #@param {type:"number"}
D = D * 1.0e-9
alpha = -3 #@param {type:"slider", min:-5, max:5, step:1}
alpha = 10**(alpha)
K = 15.9 #@param {type:"number"}
K = K * 1.0e3
d = 20 #@param {type:"number"}
d = d * 1.0e-9
T = 293 #@param {type:"number"}

# Calculate Neel relaxation time with user-defined parameters
tau_N = calculate_neel_relaxation_time(K, d, T)
# Calculate Brown relaxation time with user-defined parameters
tau_B = calculate_brown_relaxation_time(eta, D, T, rate = alpha)
# Calculate effective relaxation time using the previously calculated tau_B and tau_N
tau_eff = calculate_effective_relaxation_time(tau_B, tau_N)

if tau_N * 20 < tau_B:
    print("Neel is dominant.")
elif tau_B * 20 < tau_N:
    print("Brown is dominant.")
else:
    print("The tow relaxation is competing.")

# Assign the selected relaxation time to a variable for use in the AC Langevin model
if relaxation_time_choice == "Brown":
    selected_relaxation_time = tau_B
elif relaxation_time_choice == "Neel":
    selected_relaxation_time = tau_N
else:
    selected_relaxation_time = tau_eff

print(f"{relaxation_time_choice} Relaxation Time: tau = {selected_relaxation_time:.3g} s")

try:
# @markdown Set Drive Field and Magnetization Parameta
    Ms = 50 #@param {"type": "number"}
    Ms = Ms * 1.0e3
    if unit == "cgs":
        Ms = Ms * density * 1.0e-3
        Ms = cgs_SI(Ms, "M")
    chi0 = 3 #@param {"type": "number"}
    langevin_model1 = LangevinMagnetization(M_s=Ms, chi_0=chi0)
    H0 = 300 #@param {"type": "number"}
    H0 = H0 * 1.0e3
    if unit == "cgs":
        H0 = H0 * 1.0e-3
        H0 = cgs_SI(H0, "H")
    frequency_ac = 1 #@param {"type": "number"}
    frequency_ac = frequency_ac * 1.0e3 # Convert to Hz
    # The relaxation_time_ac param is now replaced by selected_relaxation_time

    # @markdown Set DFT Parameta
    loop = 4 #@param {"type": "number"}
    n_dft = 4096 #@param {"type": "number"}
    plot_1st_harmonic = True #@param {"type": "boolean"}
    plot_1st_harmonic = not plot_1st_harmonic
    plot_voltage = True #@param {"type": "boolean"}

    # Create an instance of AC_LangevinMagnetization using the selected relaxation time
    ac_model = DFTMagnetization(
        M_s=Ms,
        chi_0=chi0,
        frequency=frequency_ac,
        relaxation_time=selected_relaxation_time
    )

    # Simulate AC magnetization
    t, H_t, M_t = ac_model.calculate_ac_magnetization(
        H_max=H0,
        loop=loop,
        n = n_dft
    )

    # Plot Magnetization Curve
    ac_model.plot_ac_magnetization_curve(t, H_t, M_t, unit=unit)

    # Calculate Voltage
    V_t = ac_model.calculate_ac_voltage(t, H_t, M_t, plot=plot_voltage)

    # Calculate Harmonics
    H_harmonics, M_harmonics = ac_model.calculate_harmonics(t, V_t, num_harmonics=6)

    # Plot the harmonics
    ac_model.plot_harmonics(H_harmonics, M_harmonics, rid1=plot_1st_harmonic)

except ValueError as e:
    print(f"Error: {e}")


In [ ]:
# @title Calculate $A$ matrix

from tqdm.auto import tqdm

Etas = np.logspace(0, 2, 100) * 1.0e-3
Freqs =np.array([500, 600, 700, 850, 1000, 1150, 1300, 1450,
                 1600, 1800, 2000, 2300, 2600, 3000, 3400, 3800,
                 4300, 4800, 5500, 6200, 6900, 7800, 8800, 10000,
                 13000, 17000, 23000], dtype=np.float32)
A_mat = np.zeros((len(Freqs), len(Etas)))

bar = tqdm(total = len(Etas) * len(Freqs))
for i, eta in enumerate(Etas):
  tau_B = calculate_brown_relaxation_time(eta, 300e-9, T, rate = alpha)
  for j , freq in enumerate(Freqs):
    y = 2 * np.pi * freq * tau_B
    chipp = y / (1 + y**2) * chi0
    this_ACMH = DFTMagnetization(
            M_s=Ms,
            chi_0=chi0,
            frequency=freq,
            relaxation_time=tau_B)
    t, H_t, M_t = this_ACMH.calculate_ac_magnetization(
            H_max=H0,
            loop=loop,
            n = n_dft,
            text = False)
    V_t = this_ACMH.calculate_ac_voltage(t, H_t, M_t)
    H_harmonics, M_harmonics = this_ACMH.calculate_harmonics(t, V_t, num_harmonics=6)
    # A_mat[j,i] = np.log10(M_harmonics[2])
    A_mat[j,i] = M_harmonics[2]
    bar.update(1)

A_mat_ld = np.log10(A_mat)

#plot A_mat as pcolormesh
fig, ax = plt.subplots()

Etas_mesh, Freqs_mesh = np.meshgrid(Etas*1.0e3, Freqs*1.0e-3)
im = ax.pcolormesh(Etas_mesh, Freqs_mesh, A_mat_ld, cmap='plasma')

plt.ylabel('Frequency (kHz)', fontsize=16)
plt.xlabel(r'Viscosity, $\eta$ (cP)', fontsize=16)
plt.xscale('log')
plt.yscale('log')

cbar = plt.colorbar(im, ax=ax)
cbar.set_label(r"3rd harmonics, $\log_{10}(A_{3h})$", fontsize=16)


plt.show()

# save matrix
np.save('A_mat.npy', A_mat)
np.save('Etas.npy', Etas)
np.save('Freqs.npy', Freqs)


In [ ]:
#@title Estimate $\eta$ from $\vec{F}$

# import normal distribution
from scipy.stats import norm

def solve_mu_sigma(ave, std):
  C = (std / ave)**2
  sigma = np.sqrt(np.log(C+1))
  mu = np.log(ave) - (sigma**2) / 2
  return np.exp(mu), sigma

def lognorm_pdf(x, mu=0, sigma=1):
  alpha = 1 / (x * sigma * np.sqrt(2 * np.pi))
  y = (np.log(x) - np.log(mu))
  return alpha * np.exp(-y**2 / (2 * sigma**2))

eta_ave = 8e-3
eta_std = 1e-3
mu, sigma = solve_mu_sigma(eta_ave, eta_std)
eta_dist_pdf = lognorm_pdf(Etas, mu=mu, sigma=sigma)
eta_dist_pdf = eta_dist_pdf / np.max(eta_dist_pdf)

freq_b = np.dot(A_mat, eta_dist_pdf)
re_A_mat = A_mat.copy()

Add_white_noise = False #@param {"type": "boolean"}

# Add white noise
if Add_white_noise:
  freq_b = freq_b + np.random.normal(0, np.max(freq_b) * 1e-2, len(freq_b))

# freq_b = freq_b * (1 - np.heaviside(Freqs - 100e3, 1))

U, star_A, Vt = np.linalg.svd(A_mat, full_matrices=False)

# cut Rank L

Cut_Rank = False #@param {"type": "boolean"}

if Cut_Rank:
  L = 15 #@param {"type": "number"}
  U = U[:, :L]
  star_A = star_A[:L]
  Vt = Vt[:L, :]
  re_A_mat = np.dot(U, np.dot(np.diag(star_A), Vt))

D_dag = np.linalg.inv(np.diag(star_A).T)
VD_dag = np.dot(Vt.T, D_dag)
psudo_inv_A = np.dot(VD_dag, U.T)

predict_Etas = np.dot(psudo_inv_A, freq_b)

fig, ax = plt.subplots(2, 2, figsize=(22, 20))
plt.subplots_adjust(wspace=0.12)

ax[0, 0].plot(Etas*1.0e3, eta_dist_pdf, linewidth=4)
ax[0, 0].set_xlabel(r'Solvent viscosity, $\eta$ (cP)', fontsize=16)
ax[0, 0].set_ylabel('Probability Density', fontsize=16)
ax[0, 0].set_xscale('log')
ax[0, 0].grid(True)

ax[0, 1].plot(Freqs*1.0e-3, freq_b, linewidth=4)
ax[0, 1].set_xlabel('Frequency (kHz)', fontsize=16)
ax[0, 1].set_ylabel('3rd harmonics (a.u.)', fontsize=16)
ax[0, 1].set_xscale('log')
ax[0, 1].set_yscale('log')
ax[0, 1].grid(True)

ax[1, 0].plot(Etas*1.0e3, predict_Etas, linewidth=4, label='Predicted')
ax[1, 0].plot(Etas*1.0e3, eta_dist_pdf, "--", linewidth=3, label='True')
ax[1, 0].set_xlabel(r'Solvent viscosity, $\eta$ (cP)', fontsize=16)
ax[1, 0].set_ylabel('Probability Density', fontsize=16)
ax[1, 0].set_xscale('log')
ax[1, 0].legend(fontsize=16)
ax[1, 0].grid(True)

etas_mesh, Freqs_mesh = np.meshgrid(Etas*1.0e3, Freqs*1.0e-3)
im = ax[1, 1].pcolormesh(etas_mesh, Freqs_mesh, np.log10(re_A_mat), cmap='plasma')

ax[1, 1].set_ylabel('Frequency (kHz)', fontsize=16)
ax[1, 1].set_xlabel(r'Solvent viscosity, $\eta$ (cP)', fontsize=16)
ax[1, 1].set_title('Reconstructed $A$ Matrix', fontsize=16)
ax[1, 1].set_xscale('log')
ax[1, 1].set_yscale('log')

cbar = plt.colorbar(im, ax=ax[1, 1])
cbar.set_label(r"3rd harmonics, $\log_{10}(A_{3h})$", fontsize=16)

plt.show()

# Machine Learning

In [ ]:
# @title Import required files

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm
from tqdm.auto import tqdm

A_mat = np.load('A_mat.npy')
Etas = np.load('Etas.npy')
Freqs = np.load('Freqs.npy')

dim_y, dim_x = A_mat.shape

print(f"A_mat shape: {A_mat.shape}")
print(f"dim_x (length of Ds): {dim_x}")
print(f"dim_y (length of Freqs): {dim_y}")

In [ ]:
# @title Generate dataset

datasets = []
n = 10000 #@param {"type":"integer"}
missing_rate = 0.0 #@param {"type": "number"}

for _ in tqdm(range(n)):
    this_ave = np.power(10, np.random.rand()*1.699 + 0.16) * 1.0e-3
    this_std = this_ave * (np.random.rand() * 0.15 + 0.05)
    X = norm.pdf(Etas, loc=this_ave, scale=this_std)
    Y = A_mat @ X           # Calculate Y = A_mat @ X
    missing_mask = np.random.rand(*Y.shape) < missing_rate
    Y[missing_mask] = np.nan
    Y = Y / np.max(Y[~np.isnan(Y)])
    Y = np.log10(Y)
    datasets.append((X, Y)) # Append the (X, Y) pair to the list


from sklearn.model_selection import train_test_split

all_X = np.array([data[0] for data in datasets])
all_Y = np.array([data[1] for data in datasets])

X_train, X_test, Y_train, Y_test = train_test_split(all_X, all_Y, test_size=0.2, random_state=42)

print(f"Shape of all_X: {all_X.shape}")
print(f"Shape of all_Y: {all_Y.shape}")
print(f"Total NaN in Y: {np.isnan(all_Y).sum() / (all_Y.shape[0] * all_Y.shape[1]) * 100:.2f}%")
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of Y_train: {Y_train.shape}")
print(f"Shape of Y_test: {Y_test.shape}")

# Function to apply linear interpolation to each row of a 2D array
def interpolate_rows(data_array):
    interpolated_data = []
    for row in tqdm(data_array):
        s = pd.Series(row)
        s = s.interpolate(method='linear')
        s = s.ffill().bfill()
        if s.isnull().all():
            s = s.fillna(0) # Fallback: fill entirely NaN rows with 0
        interpolated_data.append(s.values)
    return np.array(interpolated_data)

Y_train_imputed = interpolate_rows(Y_train)
Y_test_imputed = interpolate_rows(Y_test)

print(f"Shape of Y_train_imputed: {Y_train_imputed.shape}")
print(f"Shape of Y_test_imputed: {Y_test_imputed.shape}")
print(f"Total NaN in Y_train_imputed: {np.isnan(Y_train_imputed).sum()}")
print(f"Total NaN in Y_test_imputed: {np.isnan(Y_test_imputed).sum()}")

In [ ]:
# @title Train Linear ML

import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(Y_train_imputed, X_train)
print("LinearRegression model has been trained successfully.")

X_test_pred = model.predict(Y_test_imputed)

mse = mean_squared_error(X_test, X_test_pred)
r2 = r2_score(X_test, X_test_pred)

print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"R-squared (R2) Score: {r2:.4f}")

plt.figure(figsize=(10, 8))
plt.scatter(X_test, X_test_pred, alpha=0.5)
plt.plot([X_test.min(), X_test.max()], [X_test.min(), X_test.max()], 'r--', label='y=x')
plt.xlabel('Actual X_test', fontsize=14)
plt.ylabel('Predicted X_test', fontsize=14)
plt.title('Actual vs. Predicted X_test', fontsize=16)
plt.grid(True)
plt.legend(fontsize=12)
plt.show()

In [ ]:
# @title Check {"run":"auto"}

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

exp_i = 4  # @param {"type":"slider","min":0,"max":10,"step":1}

example_Y = Y_test[exp_i]
actual_X = X_test[exp_i]

imputed_example_Y_series = pd.Series(example_Y).interpolate(method='linear')
imputed_example_Y = imputed_example_Y_series.values.reshape(1, -1)

predicted_X = model.predict(imputed_example_Y)[0]

predicted_X = predicted_X / np.max(predicted_X)
actual_X = actual_X / np.max(actual_X)

print(f"Predicted: {Etas[predicted_X.argmax()]*1.0e3:.2f} cP")
print(f"Actual: {Etas[actual_X.argmax()]*1.0e3:.2f} cP")

plt.figure(figsize=(12, 6))
plt.plot(Etas * 1.0e3, predicted_X, label='Predicted X', color='red', linestyle='-')
plt.plot(Etas * 1.0e3, actual_X, label='Actual X', color='blue', linestyle='--')

plt.xlabel(r'Viscosity, $\eta$ (cP)', fontsize=14)
plt.ylabel('Probability Density', fontsize=14)

plt.title('Actual vs. Predicted X for an Example', fontsize=16)
plt.xscale('log')
plt.xlim(Etas.min()*1.0e3, Etas.max()*1.0e3)
plt.grid(True)
plt.legend(fontsize=12)

plt.show()

In [ ]:
# @title Save ML

#save the model
import joblib

model_filename = 'Freq2Eta.joblib'
joblib.dump(model, model_filename)
print(f"Model saved to {model_filename}")
